# Filtro de casos favorables con LLM

Notebook corto para comparar **cómo** pedirle a una LLM que filtre el espacio muestral,
y al final ensamblar el flujo en un `dspy.Module`.

Problema:

> Probabilidad de que la suma sea mayor que 8, sabiendo que al menos un dado mostró un 5.

Resultado esperado:

- Casos posibles (condicionados): **11**
- Casos favorables (suma > 8): **5** → `(4,5) (5,4) (5,5) (5,6) (6,5)`
- Probabilidad: **5/11**

Usamos `qwen2.5:7b`.


## 1. Configurar el LM


In [2]:
import os
import importlib
import dspy
import lm_config as dspy_lm_config

from lm_config import configure_lm


os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"
os.environ["OLLAMA_MODEL"] = "ollama_chat/qwen2.5:7b"

importlib.reload(dspy_lm_config)

lm = configure_lm(max_tokens=1024)

print("Modelo configurado:")
print(lm.model)


Modelo configurado:
ollama_chat/qwen2.5:7b


## 2. Problema y condiciones


In [3]:
problema = """
Se lanzan dos dados de seis caras.

¿Cuál es la probabilidad de que la suma de los dos dados sea mayor que 8,
sabiendo que al menos uno de los dados mostró un 5?

Expresa la probabilidad como un número entre 0 y 1.
"""


In [4]:
class IdentificarCondiciones(dspy.Signature):
    """
    Extrae las dos condiciones del problema sin resolverlo.

    No calcules probabilidades.
    No enumeres casos.
    No hagas operaciones matemáticas.
    """

    problema: str = dspy.InputField()

    condicion_espacio: str = dspy.OutputField(
        desc='Condición expresada después de "sabiendo que".'
    )

    evento_favorable: str = dspy.OutputField(
        desc="Evento cuya probabilidad se quiere calcular."
    )


identificar = dspy.ChainOfThought(IdentificarCondiciones)
condiciones = identificar(problema=problema)

print("condicion_espacio:", condiciones.condicion_espacio)
print("evento_favorable:", condiciones.evento_favorable)


condicion_espacio: Sabiendo que al menos uno de los dados mostró un 5
evento_favorable: La suma de los dos dados sea mayor que 8


## 3. Espacio muestral condicionado


In [5]:
class ConstruirEspacioCondicionado(dspy.Signature):
    """
    Construye el espacio muestral condicionado para dos dados de seis caras.

    Devuelve todos los pares (dado_1, dado_2) que cumplen la condición
    proporcionada.

    Cada dado puede tomar valores entre 1 y 6.
    No incluyas pares duplicados.
    No evalúes ninguna otra condición del problema.
    """

    condicion_espacio: str = dspy.InputField(
        desc="Condición que deben cumplir los pares."
    )

    casos_posibles: list[tuple[int, int]] = dspy.OutputField(
        desc="Todos los pares únicos que cumplen la condición."
    )


construir_espacio = dspy.Predict(ConstruirEspacioCondicionado)
espacio = construir_espacio(condicion_espacio=condiciones.condicion_espacio)

print(espacio.casos_posibles)
print("n =", len(espacio.casos_posibles))


[(5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (1, 5), (2, 5), (3, 5), (4, 5), (6, 5)]
n = 11


In [6]:
# Referencia esperada (ground truth)
ESPERADOS_POSIBLES = {
    (1, 5), (2, 5), (3, 5), (4, 5), (5, 1), (5, 2),
    (5, 3), (5, 4), (5, 5), (5, 6), (6, 5),
}
ESPERADOS_FAVORABLES = {(4, 5), (5, 4), (5, 5), (5, 6), (6, 5)}


def evaluar(nombre: str, obtenidos) -> None:
    obtenidos_set = {tuple(p) for p in obtenidos}
    ok = obtenidos_set == ESPERADOS_FAVORABLES
    inventados = obtenidos_set - ESPERADOS_POSIBLES
    faltan = ESPERADOS_FAVORABLES - obtenidos_set
    sobran = obtenidos_set - ESPERADOS_FAVORABLES

    print(f"=== {nombre} ===")
    print("resultado:", sorted(obtenidos_set))
    print("correcto:", ok)
    if inventados:
        print("inventados (fuera del espacio):", sorted(inventados))
    if faltan:
        print("faltan:", sorted(faltan))
    if sobran:
        print("de más:", sorted(sobran))
    if ok:
        print(
            f"probabilidad = {len(obtenidos_set)}/{len(ESPERADOS_POSIBLES)} "
            f"= {len(obtenidos_set) / len(ESPERADOS_POSIBLES):.4f}"
        )


## 4. Variante A — filtrar toda la lista de una vez

`Predict(FiltrarCasosFavorables)`: la LLM recibe los 11 pares y debe devolver el subconjunto.


In [7]:
class FiltrarCasosFavorables(dspy.Signature):
    """
    Filtra los casos posibles y conserva únicamente aquellos
    que cumplen todas las condiciones indicadas.

    Reglas:
    - Cada resultado es una tupla (dado_1, dado_2).
    - Solo puedes devolver pares que ya existan en casos_posibles.
    - Cada par debe cumplir la condicion_espacio.
    - Cada par debe cumplir también el evento_favorable.
    - No agregues casos nuevos.
    - No dupliques resultados.
    - Si el evento dice "mayor que N", la suma debe ser estrictamente mayor que N.
    """

    condicion_espacio: str = dspy.InputField(
        desc="Condición que define el espacio muestral."
    )

    evento_favorable: str = dspy.InputField(
        desc="Evento adicional que deben cumplir los casos favorables."
    )

    casos_posibles: list[tuple[int, int]] = dspy.InputField(
        desc="Pares previamente construidos que cumplen la condicion_espacio."
    )

    casos_favorables: list[tuple[int, int]] = dspy.OutputField(
        desc=(
            "Subconjunto de casos_posibles que cumple tanto "
            "condicion_espacio como evento_favorable."
        )
    )


filtrar_favorables = dspy.Predict(FiltrarCasosFavorables)

favorables_a = filtrar_favorables(
    condicion_espacio=condiciones.condicion_espacio,
    evento_favorable=condiciones.evento_favorable,
    casos_posibles=espacio.casos_posibles,
)

print(favorables_a.casos_favorables)
evaluar("Variante A — FiltrarCasosFavorables", favorables_a.casos_favorables)


[(5, 4), (5, 5), (5, 6), (4, 5), (6, 5)]
=== Variante A — FiltrarCasosFavorables ===
resultado: [(4, 5), (5, 4), (5, 5), (5, 6), (6, 5)]
correcto: True
probabilidad = 5/11 = 0.4545


## 5. Variante B — juez par a par

La LLM no arma la lista: solo responde True/False por cada par.
El `for` garantiza que no se inventen elementos.


In [8]:
from typing import Literal
from pydantic import BaseModel


class ReglaFavorable(BaseModel):
    operacion: Literal["suma"]
    comparador: Literal[
        "mayor_que",
        "mayor_o_igual",
        "igual",
        "menor_que",
        "menor_o_igual",
    ]
    valor: int


class InterpretarEventoFavorable(dspy.Signature):
    """
    Convierte el evento favorable en una regla estructurada.

    No resuelvas el problema.
    No calcules probabilidades.
    No enumeres casos.
    """

    evento_favorable: str = dspy.InputField()
    regla: ReglaFavorable = dspy.OutputField()


interpretar_evento = dspy.Predict(InterpretarEventoFavorable)
regla = interpretar_evento(evento_favorable=condiciones.evento_favorable)
print(regla.regla)


operacion='suma' comparador='mayor_que' valor=8


In [9]:
class CumpleRegla(dspy.Signature):
    """
    Decide si el par cumple la regla.

    Pasos:
    1. Si operacion es "suma", calcula a + b.
    2. Si comparador es "mayor_que", cumple es True solo si (a + b) > valor.
    3. Responde unicamente True o False.
    """

    par: tuple[int, int] = dspy.InputField(desc="Par (a, b) a evaluar.")
    regla: ReglaFavorable = dspy.InputField(
        desc="operacion, comparador y valor umbral."
    )
    cumple: bool = dspy.OutputField(
        desc="True si el par cumple la regla; False si no."
    )


juez = dspy.ChainOfThought(CumpleRegla)

juez.demos = [
    dspy.Example(
        par=(5, 4),
        regla=ReglaFavorable(
            operacion="suma",
            comparador="mayor_que",
            valor=8,
        ),
        cumple=True,
    ).with_inputs("par", "regla"),
    dspy.Example(
        par=(1, 5),
        regla=ReglaFavorable(
            operacion="suma",
            comparador="mayor_que",
            valor=8,
        ),
        cumple=False,
    ).with_inputs("par", "regla"),
]


def aplicar_regla_llm(casos_posibles, regla):
    favorables = []
    for par in casos_posibles:
        r = juez(par=par, regla=regla)
        if r.cumple:
            favorables.append(tuple(par))
    return favorables


favorables_b = aplicar_regla_llm(espacio.casos_posibles, regla.regla)
print(favorables_b)
evaluar("Variante B — CumpleRegla par a par", favorables_b)


[(5, 4), (5, 5), (5, 6), (4, 5), (6, 5)]
=== Variante B — CumpleRegla par a par ===
resultado: [(4, 5), (5, 4), (5, 5), (5, 6), (6, 5)]
correcto: True
probabilidad = 5/11 = 0.4545


## 6. Composición de Modules (`dspy.Module`)

Hasta acá corrimos cada paso en celdas sueltas. En DSPy el patrón es encapsularlos
en un **Module** propio: en `__init__` declarás los submódulos y en `forward`
definís el pipeline.

```text
problema
  → IdentificarCondiciones
  → ConstruirEspacioCondicionado
  → FiltrarCasosFavorables
  → Python: probabilidad = favorables / posibles
```


In [10]:
class ProbabilidadCondicional(dspy.Module):
    """Pipeline: identificar → espacio → filtrar → probabilidad."""

    def __init__(self):
        super().__init__()
        self.identificar = dspy.ChainOfThought(IdentificarCondiciones)
        self.construir_espacio = dspy.Predict(ConstruirEspacioCondicionado)
        self.filtrar = dspy.Predict(FiltrarCasosFavorables)

    def forward(self, problema: str):
        cond = self.identificar(problema=problema)
        espacio = self.construir_espacio(
            condicion_espacio=cond.condicion_espacio
        )
        filtrado = self.filtrar(
            condicion_espacio=cond.condicion_espacio,
            evento_favorable=cond.evento_favorable,
            casos_posibles=espacio.casos_posibles,
        )

        casos_posibles = [tuple(p) for p in espacio.casos_posibles]
        casos_favorables = [tuple(p) for p in filtrado.casos_favorables]
        # defensivo: no contar pares inventados fuera del espacio
        casos_favorables = [p for p in casos_favorables if p in set(casos_posibles)]

        probabilidad = (
            len(casos_favorables) / len(casos_posibles) if casos_posibles else 0.0
        )

        return dspy.Prediction(
            condicion_espacio=cond.condicion_espacio,
            evento_favorable=cond.evento_favorable,
            casos_posibles=casos_posibles,
            casos_favorables=casos_favorables,
            probabilidad=probabilidad,
        )


In [11]:
solver = ProbabilidadCondicional()
pred = solver(problema=problema)

print("condicion_espacio:", pred.condicion_espacio)
print("evento_favorable:", pred.evento_favorable)
print("casos_posibles:", pred.casos_posibles)
print("casos_favorables:", pred.casos_favorables)
print(
    f"probabilidad: {len(pred.casos_favorables)}/{len(pred.casos_posibles)} "
    f"= {pred.probabilidad:.4f}"
)

evaluar("Module ProbabilidadCondicional", pred.casos_favorables)


condicion_espacio: Sabiendo que al menos uno de los dados mostró un 5
evento_favorable: La suma de los dos dados sea mayor que 8
casos_posibles: [(5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (1, 5), (2, 5), (3, 5), (4, 5), (6, 5)]
casos_favorables: [(5, 4), (5, 5), (5, 6), (4, 5), (6, 5)]
probabilidad: 5/11 = 0.4545
=== Module ProbabilidadCondicional ===
resultado: [(4, 5), (5, 4), (5, 5), (5, 6), (6, 5)]
correcto: True
probabilidad = 5/11 = 0.4545


## Conclusión

| Enfoque | Rol |
|---------|-----|
| Variante A | Probar el filtro en una sola llamada |
| Variante B | Alternativa par a par si A falla con modelos chicos |
| `ProbabilidadCondicional` | **Compose modules**: un solo `solver(problema=...)` |

La idea de DSPy: Signatures = *qué*; Modules = *cómo* y *en qué orden*.
